# Northstar Code Arena — Colab coursework

This notebook runs the Python submitted from Code Arena and returns the interactive validation result to the learning platform. Sign in to your own Google account, select the Colab runtime you need, and run the cells in order.

The pairing code is single-use and short-lived. A validation result from this notebook is labelled **Colab validation**; it is not an external Online Judge verdict.

In [ ]:
from getpass import getpass
import requests

def northstar_api_data(response):
    try:
        payload = response.json()
    except ValueError as error:
        raise RuntimeError(f"Platform returned a non-JSON response ({response.status_code})") from error
    if not response.ok or payload.get("ok") is not True:
        message = payload.get("error", {}).get("message", response.text)
        raise RuntimeError(f"Platform request failed ({response.status_code}): {message}")
    return payload["data"]

API_BASE = input("Paste the Platform API URL shown in Code Arena: " ).strip().rstrip("/")
PAIRING_CODE = getpass("Paste the single-use pairing code from Code Arena: " ).strip().upper()

claim = northstar_api_data(requests.post(
    f"{API_BASE}/api/colab/claim",
    json={"pairingCode": PAIRING_CODE},
    timeout=30,
))
RUN_ID = claim["runId"]
RUN_TOKEN = claim["accessToken"]
AUTH = {"Authorization": f"Bearer {RUN_TOKEN}"}
del PAIRING_CODE, RUN_TOKEN

bundle = northstar_api_data(requests.get(
    f"{API_BASE}/api/colab/runtime/{RUN_ID}/bundle",
    headers=AUTH,
    timeout=30,
))
print(f"Connected to activity: {bundle['activityId']}")
print(f"Validation cases loaded: {len(bundle['validation']['tests'])}")

In [ ]:
# This is the exact Python source captured from the Code Arena editor.
from IPython.display import Code, display
display(Code(bundle["sourceCode"], language="python"))

In [ ]:
# Run the learner source and the platform-provided validation harness.
import contextlib
import io
import json
import shutil
import subprocess
import traceback
import uuid

def json_safe(value):
    try:
        return json.loads(json.dumps(value))
    except (TypeError, ValueError):
        return {"repr": repr(value)}

def colab_environment_metrics():
    if not shutil.which("nvidia-smi"):
        return {"colabGpuAvailable": False}
    query = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=False,
    )
    line = query.stdout.strip().splitlines()[0] if query.stdout.strip() else ""
    if query.returncode != 0 or not line:
        return {"colabGpuAvailable": False}
    name, memory_mib = [part.strip() for part in line.split(",", 1)]
    return {"colabGpuAvailable": True, "colabGpuName": name, "gpuMemoryMiB": int(memory_mib)}

captured_stdout = io.StringIO()
captured_stderr = io.StringIO()
namespace = {
    "__name__": "__main__",
    "__northstar_tests__": bundle["validation"]["tests"],
}
execution_error = None

with contextlib.redirect_stdout(captured_stdout), contextlib.redirect_stderr(captured_stderr):
    try:
        exec(compile(bundle["sourceCode"], "student_submission.py", "exec"), namespace)
        exec(compile(bundle["validation"]["harnessCode"], "northstar_validation.py", "exec"), namespace)
    except (Exception, SystemExit):
        execution_error = traceback.format_exc()

validation = namespace.get("__northstar_validation__")
if execution_error is not None:
    validation = {"status": "error", "tests": [], "metrics": {}}
elif not isinstance(validation, dict):
    execution_error = "Validation harness did not produce __northstar_validation__."
    validation = {"status": "error", "tests": [], "metrics": {}}

metrics = validation.get("metrics", {})
if not isinstance(metrics, dict):
    metrics = {"reportedMetrics": json_safe(metrics)}
metrics.update(colab_environment_metrics())
stdout = captured_stdout.getvalue()
stderr = captured_stderr.getvalue() + (execution_error or "")
output_truncated = len(stdout) > 50000 or len(stderr) > 50000

candidate_callback = {
    "executionId": str(uuid.uuid4()),
    "status": validation.get("status", "error"),
    "tests": json_safe(validation.get("tests", [])),
    "returnValue": json_safe(validation.get("returnValue")),
    "metrics": json_safe(metrics),
    "stdout": stdout[:50000],
    "stderr": stderr[:50000],
    "outputTruncated": output_truncated,
}
callback_cache = globals().get("__northstar_callback_cache__")
if isinstance(callback_cache, dict) and callback_cache.get("runId") == RUN_ID:
    callback = callback_cache["callback"]
    print("Retrying the previously prepared callback with the same execution ID.")
else:
    callback = candidate_callback
    __northstar_callback_cache__ = {"runId": RUN_ID, "callback": callback}
print(json.dumps(callback, indent=2))
print("Validation prepared. Run the next cell to return it to Code Arena.")

In [ ]:
# Submit the prepared callback. This cell is safe to rerun after a network timeout.
callback_cache = globals().get("__northstar_callback_cache__")
if not isinstance(callback_cache, dict) or callback_cache.get("runId") != RUN_ID:
    raise RuntimeError("Run the validation cell for this paired session before submitting its result.")
callback = callback_cache["callback"]
accepted = northstar_api_data(requests.post(
    f"{API_BASE}/api/colab/runtime/{RUN_ID}/result",
    headers={**AUTH, "Content-Type": "application/json"},
    json=callback,
    timeout=30,
))
print(f"Colab validation returned to Code Arena: {accepted['result']['status']}")
print("If this request times out, rerun only this submit cell; it keeps the same execution ID.")

Return to Code Arena after the callback succeeds. If you change the source in Code Arena, create a new Colab session so that the new immutable bundle is used.